<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Tarea 2, DPO on-policy con vLLM**


###**Objetivo**

El objetivo de esta tarea es mejorar la capacidad de razonamiento matemático del modelo Qwen2.5-0.5B-Instruct aplicando Direct Preference Optimization (DPO). Los pares de preferencia se construyen con las propias respuestas del modelo (on-policy) y se verifican automáticamente.

###**Descripción**

En esta tarea se evaluó primero el modelo base sobre 500 problemas del benchmark GSM8K usando vLLM. Luego se generaron 8 respuestas por problema para 1500 problemas de entrenamiento. Un verificador automático las clasificó como correctas si la respuesta final estaba dentro de boxed (\boxed{}) y coincidía con la solución oficial. Con los problemas que tenían respuestas correctas e incorrectas se formaron 1157 pares de preferencia (chosen/rejected).

El modelo se entrenó con DPO combinado con una pérdida SFT (RPO), usando TRL y LoRA. Después se fusionaron los pesos y se volvió a evaluar en las mismas condiciones. La precisión subió de 47,2 % a 53,8 %, con menos respuestas truncadas y sin señales de degeneración. Esto demuestra que DPO on-policy puede mejorar un modelo pequeño sin anotaciones humanas ni un reward model.

**Ejemplo:**


Chosen (aceptada):
```
Mimi picked up 2 dozen seashells: 2 × 12 = 24 seashells.
Kyle found twice as many: 2 × 24 = 48 seashells.
Leigh grabbed one-third: ⅓ × 48 = 16 seashells.

Therefore, the number of seashells Leigh had is \boxed{16}.
```
Rejected (rechazada):
```
Mimi picked up 2 dozen seashells: 2 × 12 = 24 seashells.
Kyle found twice as many: 2 × 24 = 48 seashells.
Leigh grabbed one-third: 1/3 × 48 = 16 seashells.

So, Leigh had 16 seashells. The final answer is 16.
```

Se entrega el cuaderno ejecutado en el formulario: https://forms.gle/97XDqsezFFGYYkML8 hasta el 18 de octubre a horas 23:59

---

**1. Instalación**

Esto realiza la instalación de vLLM, TRL, PEFT y otras librerías con versiones fijadas para que el notebook sea reproducible. También desinstala torchao, porque su versión preinstalada provoca un conflicto con PEFT.

In [ ]:
!pip install -q vllm==0.31.0 trl==1.15.0 peft datasets matplotlib pandas
!pip uninstall -y -q torchao

**2. Configuración**

* Esto realiza la definición de todos los hiperparámetros del experimento: modelo, número de problemas, muestras, temperatura, β, learning rate y peso SFT.
* Además crea las carpetas data/ y results/.

In [ ]:
MODEL_ID     = "Qwen/Qwen2.5-0.5B-Instruct"

# Generación de pares
NUM_PROMPTS  = 1500 # problemas de GSM8K train para generar pares
SAMPLES = 8  # respuestas por problema
TEMPERATURE  = 1.0 # alta, para tener diversidad (algunas bien, otras mal)

# Evaluación
EVAL_EXAMPLES = 500 # problemas de GSM8K test (greedy)

# Entrenamiento DPO
BETA = 0.1
LR  = 2e-5
EPOCHS  = 1
SFT_WEIGHT = 1.0  # peso de la pérdida SFT sobre "chosen" (RPO). 0 = DPO puro

# vLLM
GPU_UTIL = 0.7  # fracción de VRAM para vLLM
EAGER_FLAG  = ""     # pon "--eager" si vLLM da problemas de compilación (p. ej. en T4)

PAIRS_PATH = "data/dpo_pairs.jsonl"
MERGED_DIR = "qwen2.5-0.5b-dpo-merged"

import os
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

**3. Utilidades comunes**

Esto realiza la creación de las utilidades compartidas:
* el prompt del sistema
* la normalización de números
* la extracción de la respuesta dentro de boxed{}.
* Incluye el verificador is_correct, que solo acepta respuestas terminadas, con boxed y con el número correcto

In [ ]:
%%writefile common.py
import re
import torch
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

SYSTEM_PROMPT = (
    "You are a careful math tutor. Solve the problem step by step, "
    "then write the final answer as a single number inside \\boxed{}."
)

def build_messages(question):
    return [
        {"role": "system", "content": #TU CODIGO AQUI},
        {"role": "user", "content": #TU CODIGO AQUI},
    ]

def normalize(s):
    s = s.strip().replace(",", "").replace("$", "").rstrip(".")
    try:
        x = float(s)
        return str(int(x)) if x.is_integer() else str(x)
    except ValueError:
        return s

def gold_answer(answer_field):
    # En GSM8K la respuesta final va después de "####"
    return normalize(answer_field.split("####")[-1])

BOXED = re.compile(r"\\boxed\{([^{}]*)\}")
NUM = re.compile(r"-?\d[\d,]*(?:\.\d+)?")

def extract_answer(text):
    """Devuelve (respuesta, usó_boxed)."""
    boxed = BOXED.findall(text)
    if boxed:
        nums = NUM.findall(boxed[-1])
        return (normalize(nums[-1]) if nums else normalize(boxed[-1])), True
    nums = NUM.findall(text)
    return (normalize(nums[-1]) if nums else None), False

def is_correct(text, gold, finished=True):
    """Criterio estricto: usa \\boxed{}, coincide con la respuesta real y no se cortó."""
    pred, boxed = extract_answer(text)
    return finished and boxed and pred == gold

def load_gsm8k(split):
    return load_dataset("openai/gsm8k", "main", split=split)

def vllm_dtype():
    return "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"

**4. Script de evaluación (vLLM, greedy)**

* esto realiza la evaluación del modelo sobre 500 problemas de GSM8K test, generando con vLLM en modo greedy
* calcula precisión estricta y laxa, uso de boxed, tasa de truncamiento y longitud media y lo guarda todo en un JSON

In [ ]:
%%writefile evaluate.py
import argparse, json
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from common import build_messages, extract_answer, gold_answer, is_correct, load_gsm8k, vllm_dtype

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--num_examples", type=int, default=500)
    p.add_argument("--max_tokens", type=int, default=512)
    p.add_argument("--gpu_util", type=float, default=0.7)
    p.add_argument("--eager", action="store_true")
    p.add_argument("--out", required=True)
    a = p.parse_args()

    ds = load_gsm8k("test").select(range(a.num_examples))
    tok = AutoTokenizer.from_pretrained(a.model)
    prompts = [tok.apply_chat_template(build_messages(#TU CODIGO AQUI),
                                       tokenize=False,
                                       add_generation_prompt=True)
               for q in ds["question"]]

    llm = LLM(model=a.model, dtype=vllm_dtype(), gpu_memory_utilization=a.gpu_util,
              max_model_len=2048, seed=0, enforce_eager=a.eager)
    outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=a.max_tokens))

    rows, strict, lenient, boxed_n, trunc, ntok = [], 0, 0, 0, 0, 0
    for ex, out in zip(ds, outs):
        c = out.outputs[0]
        gold = gold_answer(ex["answer"])
        pred, boxed = extract_answer(c.text)
        finished = c.finish_reason == "stop"
        ok = is_correct(c.text, gold, finished)
        strict += ok
        lenient += pred == gold
        boxed_n += boxed
        trunc += not finished
        ntok += len(c.token_ids)
        rows.append({"question": ex["question"], "gold": gold, "pred": pred,
                     "correct": bool(ok), "response": c.text})

    n = len(rows)
    metrics = {
        "model": a.model,
        "accuracy_strict": strict / n,    # boxed + correcto
        "accuracy_lenient": lenient / n,  # último número correcto
        "boxed_rate": boxed_n / n,
        "truncated_rate": trunc / n,
        "avg_tokens": ntok / n,
    }
    print(json.dumps(metrics, indent=2))
    with open(a.out, "w") as f:
        json.dump({"metrics": metrics, "samples": rows}, f, indent=2)

if __name__ == "__main__":
    main()

**5. Script de generación de pares de preferencia**

* esto realiza la generación de 8 respuestas por problema de GSM8K train y las clasifica como correctas o incorrectas
* con los problemas mixtos forma un par (chosen, rejected)
* descarta los que tienen todas las respuestas bien o todas mal

El dataset queda en formato conversacional, que DPOTrainer entiende directamente y le aplica el chat template de Qwen.

In [ ]:
%%writefile generate_pairs.py
import argparse, json, random
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from common import build_messages, gold_answer, is_correct, load_gsm8k, vllm_dtype

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--num_prompts", type=int, default=1500)
    p.add_argument("--samples", type=int, default=8)
    p.add_argument("--temperature", type=float, default=1.0)
    p.add_argument("--max_tokens", type=int, default=512)
    p.add_argument("--gpu_util", type=float, default=0.7)
    p.add_argument("--eager", action="store_true")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--out", required=True)
    a = p.parse_args()
    random.seed(a.seed)

    ds = load_gsm8k("train").shuffle(seed=a.seed).select(range(a.num_prompts))
    tok = AutoTokenizer.from_pretrained(a.model)
    prompts = [tok.apply_chat_template(build_messages(q), tokenize=False, add_generation_prompt=True)
               for q in ds["question"]]

    llm = LLM(model=a.model, dtype=vllm_dtype(), gpu_memory_utilization=a.gpu_util,
              max_model_len=2048, seed=a.seed, enforce_eager=a.eager)
    sp = SamplingParams(n=a.samples, temperature=a.temperature, top_p=0.95, max_tokens=a.max_tokens)

    outs = llm.generate(#TU CODIGO AQUI, #TU CODIGO AQUI)

    pairs, n_correct, n_total = [], 0, 0
    all_good = all_bad = 0
    for ex, out in zip(ds, outs):
        gold = gold_answer(ex["answer"])
        good, bad = [], []
        for c in out.outputs:
            ok = is_correct(c.text, gold, c.finish_reason == "stop")
            (good if ok else bad).append(c.text)
        n_correct += len(good)
        n_total += len(out.outputs)
        if not bad:
            all_good += 1
        elif not good:
            all_bad += 1
        else:
            pairs.append({
                "prompt": build_messages(ex["question"]),
                "chosen": [{"role": "assistant", "content": random.choice(#TU CODIGO AQUI)}],
                "rejected": [{"role": "assistant", "content": random.choice(#TU CODIGO AQUI)}],
            })

    with open(a.out, "w") as f:
        for r in pairs:
            f.write(json.dumps(r) + "\n")

    print(f"Precisión media de las muestras: {n_correct / n_total:.1%}")
    print(f"Problemas con todas bien: {all_good} | todas mal: {all_bad} | mixtos: {len(pairs)}")
    print(f"Pares DPO guardados: {len(pairs)} -> {a.out}")

if __name__ == "__main__":
    main()

**6. Evaluación del modelo base**

* esto realiza la medición del rendimiento inicial de Qwen2.5-0.5B antes del entrenamiento
* deberias obtener los siguientes valores:


```
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "accuracy_strict": 0.472,
  "accuracy_lenient": 0.476,
  "boxed_rate": 0.946,
  "truncated_rate": 0.048,
  "avg_tokens": 297.364
```


In [ ]:
!python evaluate.py --model {MODEL_ID} --num_examples {EVAL_EXAMPLES} --gpu_util {GPU_UTIL} {EAGER_FLAG} --out results/base.json

**7. Generar el dataset de preferencias**

Esto realiza la ejecución del script de pares: 12 000 respuestas en unos 10 minutos, con un 50,8 % de aciertos. De los 1500 problemas, 1157 son mixtos y se convierten en pares DPO.

In [ ]:
!python generate_pairs.py --model {MODEL_ID} --num_prompts {NUM_PROMPTS} --samples {SAMPLES} --temperature {TEMPERATURE} --gpu_util {GPU_UTIL} {EAGER_FLAG} --out {PAIRS_PATH}

**Vemos un par de ejemplo**

* esto realiza la carga del dataset generado y muestra la pregunta, la respuesta elegida y la rechazada de un par
* en este ejemplo, la rechazada tiene el número correcto pero no usa boxed, así que se rechaza solo por formato

In [ ]:
from datasets import load_dataset

pairs = load_dataset("json", data_files=PAIRS_PATH, split="train")
print(pairs)
ex = pairs[0]
print("PREGUNTA:\n", ex["prompt"][-1]["content"])
print("\nCHOSEN:\n", ex["chosen"][0]["content"][:1200])
print("\nREJECTED:\n", ex["rejected"][0]["content"][:1200])

**8. Entrenamiento DPO (TRL + LoRA)**

* con LoRA no hace falta cargar una copia aparte del modelo de referencia: TRL usa el mismo modelo con el adaptador desactivado como $\pi_{ref}$
* realiza el entrenamiento del modelo con DPO, usando LoRA y entrenando solo el 1,75 % de los parámetros durante 69 pasos
* el training loss deberia bajar de ~0.86 a ~0.70

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

use_bf16 = torch.cuda.get_device_capability()[0] >= 8

tokenizer = AutoTokenizer.from_pretrained(#TU CODIGO AQUI)
model = AutoModelForCausalLM.from_pretrained(
    #TU CODIGO AQUI,
    dtype=torch.bfloat16 if use_bf16 else torch.float32
)

split = pairs.train_test_split(test_size=0.05, seed=42)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", task_type="CAUSAL_LM",
)

args = DPOConfig(
    output_dir="qwen-dpo-lora",
    beta=#TU CODIGO AQUI,
    loss_type=["sigmoid", "sft"] if SFT_WEIGHT > 0 else ["sigmoid"],
    loss_weights=[1.0, SFT_WEIGHT] if SFT_WEIGHT > 0 else None,
    learning_rate=#TU CODIGO AQUI,
    num_train_epochs=#TU CODIGO AQUI,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    max_length=1024,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=42,
)

trainer = DPOTrainer(
    model=#TU CODIGO AQUI,
    args=#TU CODIGO AQUI,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tokenizer,
    peft_config=#TU CODIGO AQUI,
)
trainer.model.print_trainable_parameters()
trainer.train()

**9. Fusionar LoRA y guardar el modelo para vLLM**

* esto realiza la fusión de los pesos LoRA con el modelo base y guarda el resultado en fp16 para que vLLM pueda cargarlo directamente
* después libera la memoria de la GPU para la evaluación siguiente

In [ ]:
import gc

merged = trainer.model.merge_and_unload()
merged = merged.to(torch.bfloat16 if use_bf16 else torch.float16)
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Guardado en", MERGED_DIR)

del merged, model, trainer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM ocupada tras liberar: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

**10. Evaluación del modelo DPO**

* esto realiza la misma evaluación de la sección 4, ahora sobre el modelo entrenado y con los mismos 500 problemas
* deberias obtener los siguientes valores:


```
{
  "model": "qwen2.5-0.5b-dpo-merged",
  "accuracy_strict": 0.538,
  "accuracy_lenient": 0.544,
  "boxed_rate": 0.962,
  "truncated_rate": 0.034,
  "avg_tokens": 290.91
}
```



In [ ]:
!python evaluate.py --model {MERGED_DIR} --num_examples {EVAL_EXAMPLES} --gpu_util {GPU_UTIL} {EAGER_FLAG} --out results/dpo.json

**11. Comparación**

* esto realiza una tabla comparativa de las métricas del modelo base y del modelo DPO

In [ ]:
import json
import pandas as pd

base = json.load(open("results/base.json"))
dpo = json.load(open("results/dpo.json"))

df = pd.DataFrame([base["metrics"], dpo["metrics"]], index=["base", "dpo"]).drop(columns="model")
display(df.style.format({c: "{:.1%}" for c in df.columns if c != "avg_tokens"} | {"avg_tokens": "{:.0f}"}))

fixed = [i for i, (b, d) in enumerate(zip(base["samples"], dpo["samples"])) if not b["correct"] and d["correct"]]
broken = [i for i, (b, d) in enumerate(zip(base["samples"], dpo["samples"])) if b["correct"] and not d["correct"]]
print(f"Arreglados por DPO: {len(fixed)} , Estropeados por DPO: {len(broken)}")

Un ejemplo que el modelo base fallaba y el DPO acierta

In [ ]:
if fixed:
    i = fixed[0]
    print("PREGUNTA:\n", base["samples"][i]["question"])
    print("\nRespuesta correcta:", base["samples"][i]["gold"])
    print("\n--- BASE (pred:", base["samples"][i]["pred"], ") ---\n", base["samples"][i]["response"][:1500])
    print("\n--- DPO (pred:", dpo["samples"][i]["pred"], ") ---\n", dpo["samples"][i]["response"][:1500])

cómo son las respuestas truncadas del modelo DPO?

In [ ]:
trunc = [s for s in dpo["samples"] if "\\boxed" not in s["response"]]
print(f"Respuestas DPO sin \\boxed: {len(trunc)}")
if trunc:
    print("\nFinal de una de ellas (si se repite texto, el modelo entró en bucle):\n")
    print(trunc[0]["response"][-1200:])

**Interpreta los resultados obtenidos**

Respuesta: ...

**Prueba con BETA = 0.5 ¿cambian los resultados?**

Respuesta: ...